# NB_01 — Bronze Source Ingestion

**Project:** Nexa Commerce  
**Lakehouse:** Nexa_Lakehouse  
**Layer:** Bronze  
**Purpose:** Ingest source CSV files from Lakehouse Files into managed Delta tables.

---

## Objective

This notebook performs the initial ingestion of Nexa Commerce source-system files into
Bronze Delta tables.

The Bronze layer preserves the source data as closely as possible so that downstream
transformation and data-quality processes can work from a traceable source copy.

### Source
Lakehouse `Files` area containing CSV files generated from the Nexa Commerce synthetic
source-system dataset.

### Target
Managed Delta tables in `Nexa_Lakehouse`.

### Design Principle

**Files → PySpark → Bronze Delta Tables**

No business transformations are performed in this notebook.

The following activities are intentionally deferred to downstream layers:

- Data cleansing
- Deduplication
- Data normalization
- Currency conversion
- Business-rule transformations
- Cross-channel standardization
- Fact and dimension modeling

This allows the Bronze layer to retain both valid source data and intentional
source-quality issues for later data-quality testing.


## 1. Configuration

In [6]:
from pyspark.sql import functions as F

SOURCE_ROOT = "Files/ecommerce_fabric_data"

StatementMeta(, 68b483a5-43d7-4812-b4fb-786e1d74f63f, 8, Finished, Available, Finished, False)

## 2. Source-to-Bronze Mapping

The following configuration maps source CSV files to their corresponding Bronze Delta tables.

In [2]:
source_tables = {
    "reference/suppliers.csv": "bronze_suppliers",
    "reference/locations.csv": "bronze_locations",
    "reference/product_master.csv": "bronze_product_master",
    "reference/campaigns.csv": "bronze_campaigns",

    "sales/shopify_orders.csv": "bronze_shopify_orders",
    "sales/shopify_order_lines.csv": "bronze_shopify_order_lines",

    "sales/amazon_orders.csv": "bronze_amazon_orders",
    "sales/amazon_order_lines.csv": "bronze_amazon_order_lines",

    "sales/walmart_orders.csv": "bronze_walmart_orders",
    "sales/walmart_order_lines.csv": "bronze_walmart_order_lines",

    "sales/channeladvisor_orders.csv": "bronze_channeladvisor_orders",
    "sales/channeladvisor_order_lines.csv": "bronze_channeladvisor_order_lines",

    "purchasing/purchase_orders.csv": "bronze_purchase_orders",
    "purchasing/purchase_order_lines.csv": "bronze_purchase_order_lines",

    "inventory/inventory_daily.csv": "bronze_inventory_daily",

    "refunds/refund_events.csv": "bronze_refund_events",

    "advertising/google_ads_campaign_daily.csv": "bronze_google_ads_campaign_daily",
    "advertising/walmart_connect_weekly.csv": "bronze_walmart_connect_weekly"
}

StatementMeta(, 68b483a5-43d7-4812-b4fb-786e1d74f63f, 4, Finished, Available, Finished, False)

SyntaxError: illegal target for annotation (3418479587.py, line 1)

## 3. Bronze Ingestion

Each CSV source is read from the Lakehouse Files area and written as a managed
Delta table.

The source data is not transformed during ingestion.

In [15]:
# ============================================================
# 3. Bronze Ingestion
# ============================================================

ingestion_results = []

for source_file, target_table in source_tables.items():

    source_path = f"{SOURCE_ROOT}/{source_file}"

    try:
        print(f"Loading: {source_file}")

        # Read CSV without schema inference.
        # Bronze preserves source values as strings.
        df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "false")
            .load(source_path)
        )

        row_count = df.count()
        column_count = len(df.columns)

        # Write source data as managed Delta table
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )

        ingestion_results.append({
            "source_file": source_file,
            "target_table": target_table,
            "row_count": row_count,
            "column_count": column_count,
            "status": "PASS",
            "error": None
        })

        print(
            f"  PASS | {target_table} | "
            f"{row_count:,} rows | {column_count} columns"
        )

    except Exception as e:

        ingestion_results.append({
            "source_file": source_file,
            "target_table": target_table,
            "row_count": None,
            "column_count": None,
            "status": "FAIL",
            "error": str(e)
        })

        print(f"  FAIL | {target_table}")
        print(f"  Error: {str(e)}")


# ------------------------------------------------------------
# Display ingestion summary
# ------------------------------------------------------------

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType
)

results_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("row_count", LongType(), True),
    StructField("column_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error", StringType(), True)
])

results_df = spark.createDataFrame(
    ingestion_results,
    schema=results_schema
)

display(
    results_df.orderBy("status", "target_table")
)

StatementMeta(, cd09f4c6-7e06-4dfc-a2fd-4850e0571ec3, 17, Finished, Available, Finished, False)

Loading: reference/suppliers.csv
  PASS | bronze_suppliers | 12 rows | 5 columns
Loading: reference/locations.csv
  PASS | bronze_locations | 4 rows | 5 columns
Loading: reference/product_master.csv
  PASS | bronze_product_master | 120 rows | 10 columns
Loading: reference/campaigns.csv
  PASS | bronze_campaigns | 20 rows | 6 columns
Loading: sales/shopify_orders.csv
  PASS | bronze_shopify_orders | 1,378 rows | 11 columns
Loading: sales/shopify_order_lines.csv
  PASS | bronze_shopify_order_lines | 2,770 rows | 8 columns
Loading: sales/amazon_orders.csv
  PASS | bronze_amazon_orders | 1,640 rows | 11 columns
Loading: sales/amazon_order_lines.csv
  PASS | bronze_amazon_order_lines | 3,279 rows | 8 columns
Loading: sales/walmart_orders.csv
  PASS | bronze_walmart_orders | 1,228 rows | 11 columns
Loading: sales/walmart_order_lines.csv
  PASS | bronze_walmart_order_lines | 2,416 rows | 8 columns
Loading: sales/channeladvisor_orders.csv
  PASS | bronze_channeladvisor_orders | 754 rows | 12 c

SynapseWidget(Synapse.DataFrame, abcf8f52-858d-4a31-9b9d-64023f940304)